In [0]:
%pip install pymongo
%pip install pyyaml

dbutils.library.restartPython()

In [0]:
import time
from pymongo import MongoClient
from pymongo.errors import PyMongoError

class MongoExtractor:
    def __init__(self, database="sample_mflix"):
        uri = dbutils.secrets.get(scope="conn-db", key="cnn-mongodb-sampleflix")
        self.db = MongoClient(uri)[database]

    def extract(self, collection, modo_carga="full", campo_watermark=None,
                watermark_value=None, campos=None, batch_size=2000, max_retries=3):
        filtro = {campo_watermark: {"$gt": watermark_value}} if modo_carga == "incremental" and watermark_value else {}
        projecao = {c: 1 for c in campos} if campos else None

        for tentativa in range(max_retries):
            try:
                cursor = self.db[collection].find(filtro, projecao).batch_size(batch_size)
                lote = []
                for doc in cursor:
                    lote.append(doc)
                    if len(lote) >= batch_size:
                        yield lote
                        lote = []
                if lote:
                    yield lote
                return
            except PyMongoError:
                if tentativa == max_retries - 1:
                    raise
                time.sleep(2 ** tentativa)